# CO₂–N₂O slab: Temperature dependence study of Transmission (HITRAN · Voigt · Beer–Lambert)

Choose a **pressure**, a **path length** and a **temperature range** with sliders (SI units: Pa, m, K); the notebook then animates the
transmittance of a CO₂ + N₂O gas slab over that temperature range, plotted against **wavelength** (4.2553–4.4444 µm, i.e. 2250–2350 cm⁻¹).
**Run the cells top to bottom.**

**Contents** 1 Atmospheric layer and ranges · 2 Theory · 3 Setup · 4 Physics engine · 5 HITRAN input/output · 6 Tests · 7 Load data and study table ·
8 Interactive animation · 9 Checks on real data · 10 Export · 11 References

## 1. Which atmospheric layer, and which ranges? (the study)

**Layer: the troposphere.** Reasons, from the AFGL reference atmospheres (Anderson et al. 1986, AFGL-TR-86-0110, which tabulates six standard atmospheres):

1. **Uniform mixing ratios.** In those profiles CO₂ has a single constant mixing ratio (330 ppmv, a 1986 value), and N₂O stays close to its surface value
   (about 0.32 ppmv) through the troposphere and decreases with height in the stratosphere. A slab with *uniform* CO₂ and N₂O mole fractions
   is therefore a fair model of the troposphere, but not of the stratosphere.
2. **Conditions HITRAN is built for.** HITRAN's line parameters are referenced to 296 K and 1 atm; tropospheric temperatures and pressures bracket these.
3. **Simple line shapes.** At 0.09–1 bar the lines are Voigt profiles dominated by collisional (Lorentz) broadening; run `study_table()` in §7 after loading data to see
   the Lorentz/Doppler width ratio for your lines.

**Ranges sampled from the six AFGL atmospheres** (values read from a scanned copy and cross-checked against the ideal-gas density column of the report):

| quantity | tropospheric range | source values | slider (SI) |
|---|---|---|---|
| temperature | 194.8 – 299.7 K | coldest: tropical, 17 km (93.7 hPa); warmest: tropical, surface | 190 – 300 K, default sweep 195 – 300 K |
| pressure | 93.7 – 1018 hPa | tropical cold point … mid-latitude-winter surface (the six surface pressures are 1010–1018 hPa) | 9 000 – 102 000 Pa, default 54 000 Pa (≈ 5 km, U.S. Standard: 540.5 hPa) |
| path length | 1 cm – 10 km | laboratory cell … horizontal path in the layer | 0.01 – 10 000 m (logarithmic), default 100 m |

U.S. Standard (1976) reference points in the same tables: 0 km: 288.2 K, 1013 hPa · 5 km: 255.7 K, 540.5 hPa · 11 km: 216.8 K, 227.0 hPa.

**About the path length.** The Beer–Lambert slab needs T and P to be uniform along the path. That is natural for a *horizontal* path. A *vertical* path is not uniform:
in the U.S. Standard troposphere the temperature falls 6.5 K per km and the pressure drops by about 11 % in the first kilometre, so vertical paths longer than about 1 km need
a layered model (a later stage of this project). The range limits above are my choice, not a standard.

**Temperature and pressure are independent sliders here.** In the real atmosphere they are correlated, so a combination such as 300 K with 9 000 Pa is a what-if experiment, not a standard-atmosphere state.

**What the model still leaves out in the troposphere:** water vapour (up to roughly 0.8 % by volume near the surface in the U.S. Standard profile), clouds and aerosols, scattering, thermal emission of the gas.
Mole fractions: CO₂ 423.03 ppm (NOAA GML, Sep 2025) and N₂O 339.76 ppb (NOAA GML, Mar 2026), dry air, uniform.

**Window warning.** The N₂O ν₃ band is centred near 2224 cm⁻¹ and lies mostly below about 2260 cm⁻¹ (from the spectroscopy literature; check it against your downloaded line list with `window_content()`).
The requested window 2250–2350 cm⁻¹ therefore contains only the high-wavenumber **tail** of the N₂O band, on top of the CO₂ ν₃ P-branch. N₂O will be a weak feature here.
To see the N₂O band properly, lower `cfg.NU_MIN` to about 2180 and re-run from the setup cell.

## 2. Theory in brief

**Gas.** Ideal gas: $n = P/(k_BT)$, partial pressure $p_i = x_iP$, $n_i = x_i n$. At fixed $P$, cooling *raises* $n$.

**Line strength.** A line has position $\tilde\nu_0$ (cm$^{-1}$), lower-state energy $E''$ (cm$^{-1}$) and strength
$S$ in cm$^{-1}$/(molecule cm$^{-2}$) $=$ cm/molecule, with $\int\sigma\,d\tilde\nu=S$. HITRAN's $S$ already contains the natural isotopic
abundance, so $n$ is the *total* number density of the gas.

**Temperature scaling of $S$ (HITRAN definition):**
$$S(T)=S(T_{ref})\,\frac{Q(T_{ref})}{Q(T)}\;\frac{e^{-c_2E''/T}}{e^{-c_2E''/T_{ref}}}\;\frac{1-e^{-c_2\tilde\nu_0/T}}{1-e^{-c_2\tilde\nu_0/T_{ref}}},\qquad T_{ref}=296\,\mathrm K,\; c_2=\tfrac{hc}{k_B}=1.4388\ \mathrm{cm\,K}$$
* $Q(T)$ (partition function) grows with $T$ as more states are populated → lowers *every* line.
* $e^{-c_2E''/T}$ moves population to high-$E''$ levels → raises high-$E''$ lines, lowers low-$E''$ lines.
* The last factor is the stimulated-emission correction ($\approx1$ here since $c_2\tilde\nu_0/T\approx13$).
* There is **no universal trend**. With $Q\propto T$ a line's strength peaks at $T=c_2E''$. For an optically thin line at *fixed pressure*, $W\propto nS$, so
  $d\ln W/d\ln T=-2+c_2E''/T$: only lines with $E''>2T/c_2$ strengthen on warming.

**Widths (always HWHM, never FWHM).**
$$\gamma_D=\frac{\tilde\nu_0}{c}\sqrt{\frac{2k_BT\ln2}{m}},\qquad
\gamma_L=\Big(\tfrac{T_{ref}}{T}\Big)^{n_{air}}\big[\gamma_{air}(P-p_{self})+\gamma_{self}\,p_{self}\big]\ (\text{pressures in atm}),\qquad \tilde\nu_0'=\tilde\nu_0+\delta_{air}P/\mathrm{atm}$$
The Gaussian (Doppler) part comes from the Maxwell velocity distribution; the Lorentzian part from collision-shortened state lifetimes. They are independent, so the line shape is their convolution, the Voigt profile
$$\varphi(\Delta\tilde\nu)=\frac{\mathrm{Re}\,w(z)}{\sigma_G\sqrt{2\pi}},\quad z=\frac{\Delta\tilde\nu+i\gamma_L}{\sigma_G\sqrt2},\quad \sigma_G=\frac{\gamma_D}{\sqrt{2\ln2}}$$
with $w$ the Faddeeva function. $\int\varphi\,d\tilde\nu=1$ exactly; a finite window misses a Lorentzian tail $\approx 2\gamma_L/(\pi W)$.

**Absorption and transmission.**
$$\sigma_i=\sum_k S_k(T)\varphi_k,\quad \alpha_i=n_i\sigma_i,\quad \alpha=\alpha_{CO_2}+\alpha_{N_2O},\quad \tau=\alpha L,\quad \mathcal T=e^{-\tau},\quad I=I_0\mathcal T$$
Intensity $I$ and transmittance $\mathcal T$ are different quantities; with a flat $I_0$ they have the same shape, with a blackbody $I_0$ they do not.
**Wavelength.** Everything is computed on HITRAN's wavenumber grid and plotted against $\lambda[\mu\mathrm m]=10^4/\tilde\nu[\mathrm{cm^{-1}}]$. A source spectrum per unit wavelength is
$B_\lambda=\frac{2hc^2}{\lambda^5}\big(e^{hc/\lambda k_BT}-1\big)^{-1}$; wavelength-weighted integrals use $d\lambda=10^4\,d\tilde\nu/\tilde\nu^2$.
Diagnostics: the transmittance at a selected wavelength, the wavelength-weighted window mean of $\mathcal T$, and the equivalent width $W=\int(1-\mathcal T)\,d\lambda$ (nm).
$W$ is **not** an absorbed energy.

**Thin vs saturated lines.** Thin: $W\approx SN$ ($N=nL$). Strongly saturated Lorentzian line: $W\approx2\sqrt{SN\gamma_L}$ (only wings still absorb).

**Where it fails.** Resolution effects at large $\tau$; CO₂ far wings (a truncated Voigt ignores sub-Lorentzian wings and line mixing);
hot gas (HITRAN lacks weak hot-band lines; use HITEMP); large mole fractions (HITRAN has no CO₂/N₂O cross-broadening, air is used instead);
no emission, scattering, water vapour, or continuum.

## 3. Setup

Install once (terminal or cell): `%pip install numpy scipy matplotlib ipywidgets hitran-api` — then restart the kernel.
If widgets do not render, see the fallback at the start of §8.

In [5]:
import os, sys, json, glob, re, time, datetime
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from scipy.interpolate import CubicSpline
from scipy.special import wofz

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
    print("ipywidgets", widgets.__version__)
except ImportError:
    print("ipywidgets missing: run  %pip install -U ipywidgets  and restart the kernel (the static draw() fallback still works)")
try:
    import hapi
    print("HAPI", getattr(hapi, "__version__", "?"))
except ImportError:
    print("HAPI missing: run  %pip install hitran-api  (needed for partition sums and downloading)")
print("numpy", np.__version__)

ipywidgets 8.1.9
HAPI version: 1.3.0.0
To get the most up-to-date version please check http://hitran.org/hapi
ATTENTION: Python versions of partition sums from TIPS-2021 are now available in HAPI code

           MIT license: Copyright 2021 HITRAN team, see more at http://hitran.org. 

           If you use HAPI in your research or software development,
           please cite it using the following reference:
           R.V. Kochanov, I.E. Gordon, L.S. Rothman, P. Wcislo, C. Hill, J.S. Wilzewski,
           HITRAN Application Programming Interface (HAPI): A comprehensive approach
           to working with spectroscopic data, J. Quant. Spectrosc. Radiat. Transfer 177, 15-30 (2016)
           DOI: 10.1016/j.jqsrt.2016.03.005

           ATTENTION: This is the core version of the HITRAN Application Programming Interface.
                      For more efficient implementation of the absorption coefficient routine, 
                      as well as for new profiles, parameters and other f

In [6]:
# All illustrative settings (SI units). Edit here, then re-run from this cell downwards.
class cfg:
    # --- Spectral window (cm^-1 is what HITRAN uses internally; plots are in wavelength) ---
    NU_MIN = 2250.0           # cm^-1  -> lambda = 1e4/2250 = 4.4444 um
    NU_MAX = 2350.0           # cm^-1  -> lambda = 1e4/2350 = 4.2553 um
    NU_STEP = 0.0025          # cm^-1 grid step: must stay well below the narrowest line width, which is
                              # smallest at the LOWEST pressure of the slider (Lorentz HWHM scales with P)
    WING_CM1 = 10.0           # each line is evaluated within +/- this many cm^-1 (model choice)
    S_MIN = 1.0e-26           # drop lines weaker than this at 296 K [cm/molecule] (model choice)

    # --- Slider ranges (SI units) ---
    T_BOUNDS = (190.0, 300.0)      # K   : AFGL tropospheric envelope 194.8-299.7 K, rounded outward to 5 K
    T_DEFAULT = (195.0, 300.0)     # K   : default animated sweep (105 K)
    P_BOUNDS = (9000.0, 102000.0)  # Pa  : AFGL envelope 93.7-1018 hPa, rounded outward
    P_DEFAULT = 54000.0            # Pa  : about 5 km in the U.S. Standard (540.5 hPa, AFGL Model 6)
    P_STEP = 500.0                 # Pa
    L_BOUNDS_LOG10_M = (-2.0, 4.0)  # log10(L / m): 1 cm (laboratory cell) ... 10 km (horizontal path)
    L_DEFAULT_M = 100.0             # m, initial guess; adjust with the slider
    N_FRAMES = 101                 # frames in the animation

    # Only some frames are computed exactly ("anchors"); the others are interpolated (see rt_core).
    INTERPOLATE_FRAMES = True
    ANCHOR_DT_MAX = 10.0           # K between exact frames
    ANCHOR_DLNP_MAX = 0.07         # max change of ln(P) between exact frames (only matters if P varies)

    # --- Gas composition: dry-air mole fractions, assumed uniform in the slab ---
    X_CO2 = 423.03e-6         # NOAA GML global marine-surface monthly mean, Sep 2025
    X_N2O = 339.76e-9         # NOAA GML global monthly mean, Mar 2026
    # The remainder of the mixture is treated as non-absorbing air in this window.

    # --- Incident radiation (constant during the animation) ---
    T_SOURCE_K = 1500.0       # blackbody source temperature, illustrative lamp
    USE_FLAT_I0 = False       # True -> I0 = 1 (flat, arbitrary units)

    # --- Data ---
    DATA_DIR = "data"
    FETCH_MARGIN_CM1 = 25.0   # download this far beyond the window on each side

    SOURCES = {
        "profiles": "Anderson, G.P., Clough, S.A., Kneizys, F.X., Chetwynd, J.H., Shettle, E.P., "
                    "AFGL Atmospheric Constituent Profiles (0-120 km), AFGL-TR-86-0110, "
                    "Environmental Research Papers No. 954 (1986), https://apps.dtic.mil/sti/pdfs/ADA175173.pdf",
        "US_standard": "U.S. Standard Atmosphere, 1976 (U.S. Government Printing Office, Washington, DC)",
        "CO2": "NOAA GML, global monthly mean Sep 2025: 423.03 ppm, https://gml.noaa.gov/ccgg/trends/global.html",
        "N2O": "NOAA GML, global monthly mean Mar 2026: 339.76 ppb, https://www.gml.noaa.gov/ccgg/trends_n2o",
    }

## 4. Physics engine

Units: $\tilde\nu$ cm$^{-1}$; $S$ cm/molecule; $\varphi$ cm; $\sigma$ cm$^2$/molecule; $n$ cm$^{-3}$; $\alpha$ cm$^{-1}$; $L$ cm; widths are HWHM.

In [8]:
"""
rt_core.py - physics engine for the CO2 + N2O transmission model.

Everything here is plain NumPy/SciPy and does NOT need HITRAN data or HAPI,
so it can be unit-tested with synthetic lines (see tests/test_rt_core.py).

UNIT CONVENTIONS (HITRAN-style, CGS spectroscopy units)
  wavenumber nu ............ cm^-1
  line intensity S ......... cm^-1 / (molecule cm^-2)  == cm / molecule
  line-shape phi(nu) ....... cm            (integrates to 1 over cm^-1)
  cross section sigma ...... cm^2 / molecule   = S * phi
  number density n ......... molecule / cm^3
  absorption coeff alpha ... cm^-1         = n * sigma
  path length L ............ cm
  optical depth tau ........ dimensionless = alpha * L
  HWHM gamma ............... cm^-1  (HALF width at HALF maximum - NOT FWHM,
                                     NOT the 1/e half-width)
"""


# NumPy 2.x renamed trapz -> trapezoid; support both.
trapezoid = getattr(np, "trapezoid", None) or np.trapz

# ----------------------------------------------------------------------------
# Constants (SI values are exact since the 2019 SI redefinition)
# ----------------------------------------------------------------------------
H_PLANCK = 6.62607015e-34      # J s
C_LIGHT = 299792458.0          # m / s
K_B = 1.380649e-23             # J / K
AMU = 1.66053906660e-27        # kg (atomic mass unit; CODATA 2018 value)
ATM_PA = 101325.0              # Pa per standard atmosphere
T_REF = 296.0                  # K, HITRAN reference temperature
# second radiation constant c2 = h c / k_B, converted from m K to cm K
C2_CM_K = H_PLANCK * C_LIGHT / K_B * 100.0
LN2 = np.log(2.0)


# ----------------------------------------------------------------------------
# Data container for one gas
# ----------------------------------------------------------------------------
@dataclass
class LineList:
    """HITRAN line parameters for one molecule (all isotopologues).

    S is the HITRAN intensity at T_REF = 296 K. HITRAN's `sw` column already
    includes the natural isotopic abundance, so n_species below must be the
    TOTAL molecule number density (do not multiply by abundance again).
    """
    name: str
    M: int                      # HITRAN molecule id (CO2 = 2, N2O = 4)
    nu: np.ndarray              # cm^-1
    S: np.ndarray               # cm^-1/(molecule cm^-2) at 296 K
    E_low: np.ndarray           # cm^-1, lower-state energy
    gamma_air: np.ndarray       # cm^-1/atm, HWHM at 296 K, 1 atm
    gamma_self: np.ndarray      # cm^-1/atm, HWHM at 296 K, 1 atm
    n_air: np.ndarray           # temperature exponent of gamma_air
    delta_air: np.ndarray       # cm^-1/atm, pressure shift
    iso: np.ndarray             # local isotopologue index I (int)
    mass_amu: dict              # I -> molecular mass in amu

    def select(self, mask):
        return LineList(self.name, self.M, self.nu[mask], self.S[mask],
                        self.E_low[mask], self.gamma_air[mask],
                        self.gamma_self[mask], self.n_air[mask],
                        self.delta_air[mask], self.iso[mask], self.mass_amu)


# ----------------------------------------------------------------------------
# Ideal-gas mixture
# ----------------------------------------------------------------------------
def number_density_cm3(T, P_pa):
    """Total molecule number density n = P / (k_B T), returned in cm^-3.

    Assumes an ideal gas (valid here: ~1e5 Pa or less, 200-300 K, where the
    second virial correction is small). Partial pressure of species i is
    p_i = x_i P (Dalton) and n_i = x_i n for an ideal-gas mixture.
    """
    return P_pa / (K_B * T) * 1.0e-6        # m^-3 -> cm^-3


def dilution_warnings(x_co2, x_n2o, limit=0.01):
    """Air-broadening is used for the 'foreign' gas, which is only defensible
    if both absorbers are trace gases. Return warning strings otherwise."""
    if x_co2 < 0 or x_n2o < 0 or x_co2 + x_n2o > 1:
        raise ValueError("mole fractions must be >= 0 and sum to <= 1")
    msgs = []
    if x_co2 > limit or x_n2o > limit:
        msgs.append(
            "WARNING: x_CO2 or x_N2O exceeds %.0f%%. The model uses HITRAN "
            "air-broadening for all non-self collisions; CO2- and N2O-"
            "broadening of each other's lines is NOT available in the "
            "standard HITRAN line list, so results are an approximation "
            "and not validated in this regime." % (limit * 100))
    return msgs


# ----------------------------------------------------------------------------
# Temperature dependence of the line intensity (HITRAN definition)
#   S(T) = S(Tref) * [Q(Tref)/Q(T)]
#                  * exp(-c2 E''/T) / exp(-c2 E''/Tref)
#                  * [1 - exp(-c2 nu/T)] / [1 - exp(-c2 nu/Tref)]
# ----------------------------------------------------------------------------
def scale_intensity(S_ref, nu0, E_low, Q_ref_over_Q, T):
    """Scale HITRAN intensities from T_REF to T.

    Q_ref_over_Q : Q(T_REF)/Q(T), the partition-function ratio (per line)
    rB           : ratio of Boltzmann populations of the lower state
    rE           : ratio of stimulated-emission factors
    """
    rB = np.exp(-C2_CM_K * E_low * (1.0 / T - 1.0 / T_REF))
    rE = (-np.expm1(-C2_CM_K * nu0 / T)) / (-np.expm1(-C2_CM_K * nu0 / T_REF))
    return S_ref * Q_ref_over_Q * rB * rE


def partition_ratio_per_line(lines, T, partition_fn):
    """Q(T_REF)/Q(T) for every line (depends on the line's isotopologue).
    partition_fn(M, I, T) -> total internal partition sum (dimensionless)."""
    ratio = np.empty(len(lines.nu))
    for I in np.unique(lines.iso):
        r = partition_fn(lines.M, int(I), T_REF) / partition_fn(lines.M, int(I), T)
        ratio[lines.iso == I] = r
    return ratio


# ----------------------------------------------------------------------------
# Line widths
# ----------------------------------------------------------------------------
def doppler_hwhm(nu0, T, mass_amu):
    """Doppler HWHM [cm^-1] = (nu0/c) * sqrt(2 k_B T ln2 / m).
    (HWHM. The 1/e half-width is larger by 1/sqrt(ln2); FWHM is 2x HWHM.)"""
    m = np.asarray(mass_amu) * AMU
    return nu0 / C_LIGHT * np.sqrt(2.0 * K_B * T * LN2 / m)


def lorentz_hwhm(gamma_air, gamma_self, n_air, T, P_pa, x_self):
    """Collisional (Lorentz) HWHM [cm^-1] following the HITRAN definition:
         gamma_L = (Tref/T)^n_air * [ gamma_air (P - p_self) + gamma_self p_self ]
    with pressures in atm. 'Foreign' collisions are treated as air.
    The single exponent n_air is applied to the self part too (HITRAN
    provides no separate exponent in the standard list)."""
    P_atm = P_pa / ATM_PA
    p_self = x_self * P_atm
    return (T_REF / T) ** n_air * (gamma_air * (P_atm - p_self) + gamma_self * p_self)


# ----------------------------------------------------------------------------
# Voigt profile (Doppler Gaussian convolved with collisional Lorentzian)
# ----------------------------------------------------------------------------
def voigt_profile(dnu, gamma_D, gamma_L):
    """Normalised Voigt profile phi(dnu) [cm], with dnu = nu - nu0 [cm^-1].

    gamma_D, gamma_L are HWHM values. Gaussian sigma = gamma_D / sqrt(2 ln2).
    phi = Re[w(z)] / (sigma sqrt(2 pi)),   z = (dnu + i gamma_L) / (sigma sqrt 2)
    where w is the Faddeeva function (scipy.special.wofz).
    Integral of phi over all dnu is exactly 1.
    """
    sigma = gamma_D / np.sqrt(2.0 * LN2)
    z = (dnu + 1j * gamma_L) / (sigma * np.sqrt(2.0))
    return wofz(z).real / (sigma * np.sqrt(2.0 * np.pi))


# ----------------------------------------------------------------------------
# Cross section and absorption coefficient
# ----------------------------------------------------------------------------
def species_cross_section(grid, lines, T, P_pa, x_self, partition_fn,
                          wing_cm1=10.0, s_min=0.0):
    """sigma(nu) [cm^2/molecule] = sum_i S_i(T) * phi_i(nu), natural abundance
    included via HITRAN's S. Each line is evaluated only within +/- wing_cm1
    of its (pressure-shifted) centre: this truncation is a MODEL CHOICE
    (it drops the far Lorentzian wings and ignores sub-Lorentzian behaviour).
    """
    P_atm = P_pa / ATM_PA
    keep = ((lines.S >= s_min) & (lines.nu > grid[0] - wing_cm1)
            & (lines.nu < grid[-1] + wing_cm1))
    L = lines.select(keep)
    sigma = np.zeros_like(grid)
    if len(L.nu) == 0:
        return sigma
    S_T = scale_intensity(L.S, L.nu, L.E_low,
                          partition_ratio_per_line(L, T, partition_fn), T)
    mass = np.array([L.mass_amu[int(i)] for i in L.iso])
    nu_c = L.nu + L.delta_air * P_atm             # HITRAN pressure shift
    gD = doppler_hwhm(nu_c, T, mass)
    gL = lorentz_hwhm(L.gamma_air, L.gamma_self, L.n_air, T, P_pa, x_self)
    lo = np.searchsorted(grid, nu_c - wing_cm1)
    hi = np.searchsorted(grid, nu_c + wing_cm1)
    for k in range(len(nu_c)):
        if hi[k] > lo[k]:
            sigma[lo[k]:hi[k]] += S_T[k] * voigt_profile(
                grid[lo[k]:hi[k]] - nu_c[k], gD[k], gL[k])
    return sigma


def absorption_coefficients(grid, co2_lines, n2o_lines, T, P_pa, x_co2, x_n2o,
                            partition_fn, wing_cm1=10.0, s_min=0.0):
    """Return (alpha_CO2, alpha_N2O) in cm^-1.
    alpha_i = n_i * sigma_i,  n_i = x_i * P/(k_B T).
    The two are additive: alpha_total = alpha_CO2 + alpha_N2O."""
    n_tot = number_density_cm3(T, P_pa)
    a_co2 = x_co2 * n_tot * species_cross_section(
        grid, co2_lines, T, P_pa, x_co2, partition_fn, wing_cm1, s_min)
    a_n2o = x_n2o * n_tot * species_cross_section(
        grid, n2o_lines, T, P_pa, x_n2o, partition_fn, wing_cm1, s_min)
    return a_co2, a_n2o


# ----------------------------------------------------------------------------
# Beer-Lambert transmission
# ----------------------------------------------------------------------------
def beer_lambert(alpha_total, L_cm, I0):
    """tau = alpha L ; transmittance = exp(-tau) ; I = I0 * transmittance.
    Valid for a uniform, non-scattering, non-emitting slab and radiation
    that is narrow compared with the line structure (monochromatic)."""
    tau = alpha_total * L_cm
    trans = np.exp(-tau)
    return tau, trans, I0 * trans


def integrated_tau(grid, alpha, L_cm):
    """Integral of tau(nu) d nu  [cm^-1]  (a spectrally integrated optical depth).
    NOT an absorbed energy: that would need I0(nu) weighting and saturation."""
    return trapezoid(alpha * L_cm, grid)


def equivalent_width(grid, trans):
    """W = integral of (1 - T(nu)) d nu  [cm^-1]; spectral width of a fully
    opaque band that would absorb the same amount of a FLAT incident spectrum."""
    return trapezoid(1.0 - trans, grid)


def line_strength_vs_T(lines, k, Ts, partition_fn):
    """S(T) [cm/molecule] for line index k at each temperature in Ts."""
    one = lines.select(np.arange(len(lines.nu)) == k)
    out = []
    for T in Ts:
        qr = partition_ratio_per_line(one, T, partition_fn)
        out.append(float(scale_intensity(one.S, one.nu, one.E_low, qr, T)[0]))
    return np.array(out)


# ----------------------------------------------------------------------------
# Incident spectrum
# ----------------------------------------------------------------------------
def planck_per_cm1(nu_cm1, T_K):
    """Planck spectral radiance per unit wavenumber, W m^-2 sr^-1 (cm^-1)^-1.
    B = 2 h c^2 nu^3 / (exp(h c nu / k T) - 1) with nu in m^-1, times
    100 m^-1 per cm^-1 to convert the per-wavenumber unit."""
    nu_m = np.asarray(nu_cm1) * 100.0
    x = H_PLANCK * C_LIGHT * nu_m / (K_B * T_K)
    return 2.0 * H_PLANCK * C_LIGHT ** 2 * nu_m ** 3 / np.expm1(x) * 100.0


def planck_per_um(lam_um, T_K):
    """Planck spectral radiance per unit WAVELENGTH, W m^-2 sr^-1 um^-1:
    B = 2 h c^2 / lambda^5 / (exp(h c / (lambda k T)) - 1)   (lambda in m), times 1e-6 m/um."""
    lam = np.asarray(lam_um, dtype=float) * 1.0e-6
    x = H_PLANCK * C_LIGHT / (lam * K_B * T_K)
    return 2.0 * H_PLANCK * C_LIGHT ** 2 / lam ** 5 / np.expm1(x) * 1.0e-6


def make_incident_spectrum(grid_nu, flat, T_source):
    """I0 on the wavenumber grid, expressed per unit wavelength (plots are in wavelength).
    Returns (I0, unit string). flat=True gives I0 = 1 (arbitrary units)."""
    if flat:
        return np.ones_like(grid_nu), "arbitrary units (FLAT: I0 = 1)"
    return (planck_per_um(1.0e4 / grid_nu, T_source),
            "W m$^{-2}$ sr$^{-1}$ $\\mu$m$^{-1}$ (blackbody %.0f K)" % T_source)


def integrate_dlambda_um(grid_nu, y):
    """Integral of y over WAVELENGTH, in um, for y sampled on an increasing wavenumber grid:
    int y d(lambda) = int y * (1e4 / nu^2) d(nu)   (lambda[um] = 1e4 / nu[cm^-1])."""
    return trapezoid(np.asarray(y) * 1.0e4 / np.asarray(grid_nu) ** 2, grid_nu)


def window_mean_um(grid_nu, y):
    """Wavelength-weighted mean of y over the window (weights d(lambda))."""
    return integrate_dlambda_um(grid_nu, y) / integrate_dlambda_um(grid_nu, np.ones_like(grid_nu))


# ----------------------------------------------------------------------------
# US Standard Atmosphere 1976, troposphere (0-11 km geopotential height)
# ----------------------------------------------------------------------------
def isa_troposphere(h_m):
    """Temperature [K] and pressure [Pa] in the ISA/US-1976 troposphere:
    T = 288.15 K - 0.0065 K/m * h ;  P = 101325 Pa * (T/288.15)^(g0 M0 / R* L)."""
    g0, M0, R_star, lapse = 9.80665, 0.0289644, 8.31432, 0.0065
    T = 288.15 - lapse * np.asarray(h_m, dtype=float)
    P = 101325.0 * (T / 288.15) ** (g0 * M0 / (R_star * lapse))
    return T, P


# ----------------------------------------------------------------------------
# Many animation frames from few exact calculations
# ----------------------------------------------------------------------------
def choose_anchors(Ts, Ps, dT_max=10.0, dlnP_max=0.07):
    """Indices of frames that get an EXACT spectrum calculation. Frames must be
    sorted by increasing T. Consecutive anchors are at most dT_max kelvin apart
    and at most dlnP_max apart in ln(P) (matters when P varies along the sweep).
    The first and last frame are always anchors."""
    Ts, Ps = np.asarray(Ts, float), np.asarray(Ps, float)
    n = len(Ts)
    if n < 3:
        return np.arange(n)
    idx = [0]
    for i in range(1, n):
        j = idx[-1]
        if abs(Ts[i] - Ts[j]) > dT_max or abs(np.log(Ps[i] / Ps[j])) > dlnP_max:
            idx.append(i - 1 if i - 1 > j else i)
    if idx[-1] != n - 1:
        idx.append(n - 1)
    return np.array(sorted(set(idx)))


def interpolate_between_anchors(Ts, anchor_idx, anchor_values, rel_floor=1e-12):
    """Fill in the non-anchor frames by a cubic spline of ln(alpha) versus T.

    ln(alpha) is used because line strengths change roughly exponentially with
    temperature (Boltzmann factor), which a spline of ln(alpha) follows much
    better than a straight line in alpha. Values below rel_floor * max are
    treated as zero. This is an APPROXIMATION whose error must be checked
    (tests/test_rt_core.py and check_interpolation() in the notebook).
    anchor_values: array (n_anchor, n_grid). Returns array (len(Ts), n_grid)."""
    Ts = np.asarray(Ts, float)
    A = np.asarray(anchor_values, float)
    anchor_idx = np.asarray(anchor_idx)
    if len(anchor_idx) == len(Ts):
        return A.copy()
    peak = A.max()
    if peak <= 0:
        return np.zeros((len(Ts), A.shape[1]))
    floor = rel_floor * peak
    spline = CubicSpline(Ts[anchor_idx], np.log(np.maximum(A, floor)), axis=0)
    out = np.exp(spline(Ts))
    out[out < 2.0 * floor] = 0.0
    out[anchor_idx] = A                      # anchors stay exactly as computed
    return out


def envelope(x, y, n_bins=1500):
    """Reduce a long curve to 2*n_bins points (min and max of each bin) for
    plotting. Keeps every narrow peak and valley that a plain every-k-th-point
    thinning would miss. Output length is constant, so artists can be updated."""
    n = len(x)
    m = int(np.ceil(n / n_bins))
    if m <= 1:
        return np.asarray(x), np.asarray(y)
    pad = n_bins * m - n
    xp = np.pad(np.asarray(x, float), (0, pad), mode="edge").reshape(n_bins, m)
    yp = np.pad(np.asarray(y, float), (0, pad), mode="edge").reshape(n_bins, m)
    imin, imax = yp.argmin(axis=1), yp.argmax(axis=1)
    first, second = np.minimum(imin, imax), np.maximum(imin, imax)
    r = np.arange(n_bins)
    xs, ys = np.empty(2 * n_bins), np.empty(2 * n_bins)
    xs[0::2], ys[0::2] = xp[r, first], yp[r, first]
    xs[1::2], ys[1::2] = xp[r, second], yp[r, second]
    return xs, ys

## 5. HITRAN input/output and downloading

Lines are read from HITRAN's fixed-width `.par` format (HAPI's `.data` files use the same format), so you can either download with HAPI
**or** download manually from hitran.org (Data Access → Line-by-line, CO₂ and N₂O, all isotopologues, about 2225–2375 cm⁻¹, `.par`) and save as
`data/CO2_2225_2375.par` and `data/N2O_2225_2375.par`. HAPI is always used for isotopologue masses and the partition sums.
This notebook contains **no API key**; how HAPI 1.3.x picks the key up is not something I could verify, so check HAPI's page
(https://hitran.org/hapi/) if the download fails, and use the manual route as the fallback.

In [10]:
"""
py - read HITRAN line lists and talk to HAPI.

* Lines are read by our own parser from HITRAN's fixed-width 160-character
  .par format, so they can come from HAPI's fetch (.data files in the same
  format) OR from a manual download on hitran.org (.par files).
* HAPI is used only for the isotopologue table (masses, abundances) and for
  the partition sums Q(T) (TIPS, computed locally, no internet needed).
"""



# 0-based slice positions of the fields we need in a 160-char .par line
_SLICES = dict(M=(0, 2), I=(2, 3), nu=(3, 15), S=(15, 25), A=(25, 35),
               gamma_air=(35, 40), gamma_self=(40, 45), E_low=(45, 55),
               n_air=(55, 59), delta_air=(59, 67))


def _to_float(txt):
    """Float parser tolerant of Fortran 'D' exponents and HITRAN's odd '2.700-164'."""
    t = txt.strip().replace("D", "E")
    try:
        return float(t)
    except ValueError:
        m = re.fullmatch(r"(\d\.\d{3})-(\d{3})", t)
        if m:
            return float(m.group(1) + "E-" + m.group(2))
        raise


def _iso_index(ch):
    """Local isotopologue id: '1'-'9', '0' -> 10, 'A' -> 11, 'B' -> 12 ..."""
    if ch == "0":
        return 10
    if ch.isdigit():
        return int(ch)
    return 11 + ord(ch) - ord("A")


def parse_par_files(paths, M_expected):
    """Parse .par / .data files into column arrays (dict)."""
    cols = {k: [] for k in _SLICES}
    for path in paths:
        with open(path, "r") as fh:
            for line in fh:
                if len(line.rstrip("\n")) < 67:
                    continue
                if int(line[0:2]) != M_expected:
                    continue
                for key, (a, b) in _SLICES.items():
                    seg = line[a:b]
                    cols[key].append(_iso_index(seg) if key == "I"
                                     else int(seg) if key == "M" else _to_float(seg))
    return {k: np.array(v) for k, v in cols.items()}


def repair_missing(cols):
    """Documented policy for unusable broadening parameters (<= 0 or NaN).

    HITRAN uses placeholder values where a parameter is not available. Rather
    than invent numbers, missing values are replaced by DATA-DERIVED estimates
    (median of valid lines of the same isotopologue, else of the same gas),
    and the number of replaced lines is returned so it is reported openly.
    gamma_self falls back to the (repaired) gamma_air; because x_self is
    ~1e-4 the effect on the width is of order x_self * |gamma_self - gamma_air|.
    """
    report = {}
    for key in ("gamma_air", "n_air"):
        bad = ~(cols[key] > 0)
        report[key] = int(bad.sum())
        for I in np.unique(cols["I"][bad]) if bad.any() else []:
            sel = cols["I"] == I
            good = sel & ~bad
            ref = (np.median(cols[key][good]) if good.any()
                   else np.median(cols[key][~bad]))
            cols[key][sel & bad] = ref
    bad = ~(cols["gamma_self"] > 0)
    report["gamma_self"] = int(bad.sum())
    cols["gamma_self"][bad] = cols["gamma_air"][bad]
    return report


def iso_table(M):
    """{local I: (global_id, name, abundance, mass_amu)} from HAPI's ISO table."""
    import hapi
    out = {}
    for (m, I), v in hapi.ISO.items():
        if m == M:
            out[I] = (v[hapi.ISO_INDEX["id"]], v[hapi.ISO_INDEX["iso_name"]],
                      v[hapi.ISO_INDEX["abundance"]], v[hapi.ISO_INDEX["mass"]])
    return out


def hapi_partition_fn(M, I, T):
    """Total internal partition sum Q(M, I, T) from HAPI's built-in TIPS.
    (Which TIPS edition is used depends on the installed HAPI version.)"""
    import hapi
    q = hapi.partitionSum(M, I, float(T))
    if isinstance(q, tuple):
        q = q[1]
    return float(np.atleast_1d(q)[0])


def load_linelist(name, M, data_dir="data"):
    """Load all HITRAN files for one gas. Looks for <name>*.data and <name>*.par."""
    paths = sorted(glob.glob(os.path.join(data_dir, name + "*.data"))
                   + glob.glob(os.path.join(data_dir, name + "*.par")))
    if not paths:
        raise FileNotFoundError(
            "No %s line list in '%s'. Run fetch_hitran.py, or download a .par "
            "file from hitran.org and name it e.g. %s_2155_2425.par."
            % (name, data_dir, name))
    cols = parse_par_files(paths, M)
    if len(cols["nu"]) == 0:
        raise ValueError("Files %s contain no lines for molecule id %d" % (paths, M))
    report = repair_missing(cols)
    masses = {I: v[3] for I, v in iso_table(M).items()}
    lines = LineList(name, M, cols["nu"], cols["S"], cols["E_low"],
                     cols["gamma_air"], cols["gamma_self"], cols["n_air"],
                     cols["delta_air"], cols["I"].astype(int), masses)
    return lines, report, paths

In [11]:
MOLECULES = (("CO2", 2), ("N2O", 4))      # HITRAN molecule ids

def fetch_hitran():
    # Download CO2 and N2O line lists with HAPI and write data/data_manifest.json.
    try:
        import hapi
    except ImportError:
        raise RuntimeError("HAPI not installed. Run: %pip install hitran-api")
    os.makedirs(cfg.DATA_DIR, exist_ok=True)
    hapi.db_begin(cfg.DATA_DIR)
    numin, numax = cfg.NU_MIN - cfg.FETCH_MARGIN_CM1, cfg.NU_MAX + cfg.FETCH_MARGIN_CM1
    hapi.VARIABLES["DISPLAY_FETCH_URL"] = True
    manifest = {"retrieval_date": datetime.date.today().isoformat(), "source": "HITRANonline via HAPI",
                "hapi_version": getattr(hapi, "__version__", "unknown"),
                "wavenumber_range_cm-1": [numin, numax],
                "units": {"nu": "cm-1", "S": "cm-1/(molecule cm-2) at 296 K, abundance-weighted",
                          "gamma": "cm-1/atm HWHM at 296 K", "E_low": "cm-1"}, "gases": {}}
    for name, M in MOLECULES:
        iso = iso_table(M)
        table = "%s_%d_%d" % (name, round(numin), round(numax))
        try:
            hapi.fetch_by_ids(table, [v[0] for v in iso.values()], numin, numax)
        except Exception as exc:
            raise RuntimeError("Download failed for %s: %s\nUse the manual .par route described above." % (name, exc))
        manifest["gases"][name] = {"hitran_molecule_id": M, "table": table,
            "isotopologues": {str(I): {"global_id": v[0], "name": v[1], "abundance": v[2], "mass_amu": v[3]}
                              for I, v in iso.items()}}
    for name, M in MOLECULES:
        lines, report, paths = load_linelist(name, M, cfg.DATA_DIR)
        manifest["gases"][name].update(n_lines=int(len(lines.nu)), files=[os.path.basename(p) for p in paths],
                                       parameters_replaced_by_data_derived_estimate=report)
        print("%s: %d lines; replaced parameters: %s" % (name, len(lines.nu), report))
    with open(os.path.join(cfg.DATA_DIR, "data_manifest.json"), "w") as fh:
        json.dump(manifest, fh, indent=2)
    print("Wrote", os.path.join(cfg.DATA_DIR, "data_manifest.json"))

## 6. Tests

These 28 tests use **synthetic** lines and a toy partition function; they test the equations and the code, not HITRAN and not the real spectrum.

In [13]:
def toy_Q(M, I, T):
    return T


def make_lines(nu, S, E=0.0, g_air=0.07, g_self=0.07, n=0.7, delta=0.0, M=2, mass=44.0):
    nu = np.atleast_1d(np.asarray(nu, float))
    k = len(nu)
    full = lambda v: np.full(k, v, float) if np.isscalar(v) else np.asarray(v, float)
    return LineList("toy", M, nu, full(S), full(E), full(g_air), full(g_self),
                       full(n), full(delta), np.ones(k, int), {1: mass})


# ---------------------------------------------------------------- line shape
def _norm_with_tail(gD, gL, W, step):
    x = np.arange(-W, W + step, step)
    area = trapezoid(voigt_profile(x, gD, gL), x)
    return area, area + 2 * gL / (np.pi * W)       # + analytic Lorentzian tail


def test_voigt_normalisation():
    # Raw integral over +/-W misses the Lorentzian tail ~ 2 gL/(pi W);
    # adding that tail analytically must give 1 to high accuracy.
    for gD, gL in [(0.002, 0.04), (0.01, 0.001), (0.01, 0.0), (0.002, 0.2)]:
        raw, corrected = _norm_with_tail(gD, gL, W=2000.0, step=min(gD, max(gL, 1e-3)) / 8)
        assert abs(corrected - 1.0) < 2e-5, (gD, gL, raw, corrected)
        assert raw <= 1.0 + 1e-9


def test_voigt_limits_and_hwhm_convention():
    # pure Gaussian: peak 1/(sigma sqrt(2 pi)); value at dnu = gD must be half the peak (HWHM!)
    gD = 0.01
    peak = voigt_profile(0.0, gD, 0.0)
    assert abs(voigt_profile(gD, gD, 0.0) / peak - 0.5) < 1e-12
    sigma = gD / np.sqrt(2 * np.log(2))
    assert abs(peak - 1 / (sigma * np.sqrt(2 * np.pi))) < 1e-9 * peak
    # pure Lorentzian: peak 1/(pi gL); half maximum at dnu = gL
    gL = 0.05
    p = voigt_profile(0.0, 1e-7, gL)
    assert abs(p - 1 / (np.pi * gL)) / p < 1e-4
    assert abs(voigt_profile(gL, 1e-7, gL) / p - 0.5) < 1e-4


def test_doppler_width_scaling_and_value():
    a = doppler_hwhm(2349.0, 250.0, 44.0)
    b = doppler_hwhm(2349.0, 1000.0, 44.0)
    assert abs(b / a - 2.0) < 1e-12                       # proportional to sqrt(T)
    ref = 2349.0 / C_LIGHT * np.sqrt(2 * 1.380649e-23 * 250 * np.log(2) / (44.0 * 1.66053906660e-27))
    assert abs(a - ref) / ref < 1e-12


# ------------------------------------------------------- intensity scaling
def test_intensity_scaling_identity_and_formula():
    S = scale_intensity(1e-20, 2300.0, 500.0, 1.0, T_REF)
    assert abs(S - 1e-20) < 1e-30                          # T = T_ref -> unchanged
    T, nu, E, S0 = 230.0, 2224.0, 300.0, 2e-19
    c2 = 1.438776877
    expected = (S0 * (T_REF / T) * np.exp(-c2 * E / T) / np.exp(-c2 * E / T_REF)
                * (1 - np.exp(-c2 * nu / T)) / (1 - np.exp(-c2 * nu / T_REF)))
    got = scale_intensity(S0, nu, E, toy_Q(0, 0, T_REF) / toy_Q(0, 0, T), T)
    assert abs(got - expected) / expected < 1e-8


def test_c2_value():
    assert abs(C2_CM_K - 1.438776877) < 1e-8


def test_temperature_trends_differ_between_transitions():
    # Low-E'' line weakens on heating, high-E'' line strengthens: no universal trend.
    low = [scale_intensity(1e-19, 2300.0, 0.0, T_REF / T, T) for T in (250, 296, 400)]
    high = [scale_intensity(1e-19, 2300.0, 1500.0, T_REF / T, T) for T in (250, 296, 400)]
    assert low[0] > low[1] > low[2]
    assert high[0] < high[1] < high[2]


def test_stimulated_emission_rayleigh_jeans_limit():
    rE = scale_intensity(1.0, 1e-4, 0.0, 1.0, 200.0)   # nu -> 0: ratio -> T_ref/T
    assert abs(rE - T_REF / 200.0) < 1e-6


# -------------------------------------------------------------- broadening
def test_pressure_and_temperature_dependence_of_lorentz_width():
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 101325.0, 0.0)
    assert abs(g - 0.07) < 1e-12                           # 1 atm, 296 K, no self
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 0.5 * 101325.0, 0.0)
    assert abs(g - 0.035) < 1e-12                          # linear in P
    g = lorentz_hwhm(0.07, 0.09, 0.7, 200.0, 101325.0, 0.0)
    assert abs(g - 0.07 * (296 / 200) ** 0.7) < 1e-12
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 101325.0, 0.1)   # self part mixed in
    assert abs(g - (0.07 * 0.9 + 0.09 * 0.1)) < 1e-12


# ------------------------------------------------- units / sum rule / gas law
def test_loschmidt_number_density():
    n = number_density_cm3(273.15, 101325.0) * 1e6      # back to m^-3
    assert abs(n - 2.686780111e25) / 2.686780111e25 < 1e-6


def test_line_strength_sum_rule():
    # integral of sigma d nu over an isolated line must equal S(T) [cm/molecule]
    nu0, S0, W = 2300.0, 1e-18, 400.0
    grid = np.arange(nu0 - W, nu0 + W, 0.005)
    lines = make_lines([nu0], [S0], E=200.0)
    sig = species_cross_section(grid, lines, 250.0, 0.5 * 101325, 0.0, toy_Q, wing_cm1=W)
    S_T = scale_intensity(S0, nu0, 200.0, T_REF / 250.0, 250.0)
    gL = lorentz_hwhm(0.07, 0.07, 0.7, 250.0, 0.5 * 101325, 0.0)
    area = trapezoid(sig, grid) + S_T * 2 * gL / (np.pi * W)
    assert abs(area - S_T) / S_T < 2e-5


# -------------------------------------------------------- Beer-Lambert etc.
def _toy_mixture(x_co2=4e-4, x_n2o=3.4e-7, **kw):
    grid = np.arange(2290.0, 2310.0, 0.005)
    co2 = make_lines([2295.0, 2300.0, 2304.5], [3e-18, 1e-18, 2e-18], E=[50, 300, 900], M=2)
    n2o = make_lines([2299.0, 2302.0], [1e-18, 5e-19], E=[100, 400], M=4, mass=44.0)
    a1, a2 = absorption_coefficients(grid, co2, n2o, 255.0, 54020.0, x_co2, x_n2o, toy_Q,
                                        wing_cm1=10.0, **kw)
    return grid, a1, a2


def test_beer_lambert_consistency_and_bounds():
    grid, a1, a2 = _toy_mixture()
    I0 = np.linspace(1.0, 2.0, len(grid))
    tau, T, I = beer_lambert(a1 + a2, 1.0e4, I0)
    assert np.allclose(T, np.exp(-(a1 + a2) * 1.0e4))
    assert np.all((T >= 0) & (T <= 1))
    assert np.allclose(I, I0 * T)
    m = T > 0
    assert np.allclose(tau[m], -np.log(T[m]), rtol=1e-9, atol=1e-12)


def test_zero_absorber_gives_unit_transmission():
    grid, a1, a2 = _toy_mixture(x_co2=0.0, x_n2o=0.0)
    _, T, _ = beer_lambert(a1 + a2, 1e6, np.ones_like(grid))
    assert np.all(T == 1.0)


def test_low_abundance_and_short_path_limit():
    grid, a1, a2 = _toy_mixture(x_co2=1e-12, x_n2o=1e-12)
    tau, T, _ = beer_lambert(a1 + a2, 1.0, np.ones_like(grid))
    assert np.max(np.abs((1 - T) - tau)) < 1e-12           # 1 - exp(-tau) ~ tau


def test_optically_thick_limit():
    grid, a1, a2 = _toy_mixture(x_co2=0.2)
    _, T, _ = beer_lambert(a1 + a2, 1.0e6, np.ones_like(grid))
    assert T.min() < 1e-10


def test_mixture_additivity_and_linearity_in_mole_fraction():
    # with gamma_self == gamma_air the widths do not depend on x, so alpha is exactly linear in x
    g1, a1, _ = _toy_mixture(x_co2=2e-4)
    g2, a2, _ = _toy_mixture(x_co2=4e-4)
    assert np.allclose(2 * a1, a2, rtol=1e-12)
    # additivity: each gas computed ALONE must equal its component in the mixture,
    # so that alpha_total = alpha_CO2 + alpha_N2O with no cross-coupling in this model
    grid, c, n = _toy_mixture()
    _, c_alone, _ = _toy_mixture(x_n2o=0.0)
    _, _, n_alone = _toy_mixture(x_co2=0.0)
    assert np.allclose(c, c_alone, rtol=1e-12, atol=0)
    assert np.allclose(n, n_alone, rtol=1e-12, atol=0)
    tau_total = (c + n) * 1.0e4
    tau_sep = c * 1.0e4 + n * 1.0e4
    assert np.allclose(tau_total, tau_sep, rtol=1e-12, atol=0)


# --------------------------------------------------------- resolution tests
def test_resolution_convergence():
    rng = np.random.default_rng(1)
    nu = np.sort(rng.uniform(2295, 2305, 25))
    lines = make_lines(nu, rng.uniform(1e-19, 3e-18, 25), E=rng.uniform(0, 1500, 25))

    def integrated(step):
        grid = np.arange(2290.0, 2310.0, step)
        sig = species_cross_section(grid, lines, 255.0, 54020.0, 4e-4, toy_Q, wing_cm1=10.0)
        return trapezoid(sig, grid)
    ref = integrated(0.00125)
    err_fine = abs(integrated(0.005) - ref) / ref
    err_coarse = abs(integrated(0.1) - ref) / ref
    assert err_fine < 1e-4
    assert err_coarse > err_fine                           # error shrinks as resolution improves


# ----------------------------------------------------------- source and ISA
def test_planck_peak_follows_wien_law_in_wavenumber():
    T = 1500.0
    nu = np.arange(500.0, 8000.0, 0.5)
    peak = nu[np.argmax(planck_per_cm1(nu, T))]
    assert abs(peak - 2.8214393721 * T / C2_CM_K) < 1.0


def test_isa_troposphere_matches_tabulated_tropopause():
    T, P = isa_troposphere(11000.0)
    assert abs(T - 216.65) < 1e-9
    assert abs(P - 22632.1) / 22632.1 < 1e-3               # published table value
    T0, P0 = isa_troposphere(0.0)
    assert abs(T0 - 288.15) < 1e-12 and abs(P0 - 101325.0) < 1e-9


def test_dilution_warning():
    assert dilution_warnings(4e-4, 3e-7) == []
    assert len(dilution_warnings(0.2, 3e-7)) == 1


# ------------------------------------------- many frames from few exact ones
def _hard_lines(seed=3, n=150):
    # deliberately hard: huge spread of lower-state energies (up to 3000 cm^-1) and strengths
    rng = np.random.default_rng(seed)
    nu = np.sort(rng.uniform(2255.0, 2345.0, n))
    f = lambda v: np.full(n, v)
    return LineList("toy", 2, nu, 10 ** rng.uniform(-22, -18, n), rng.uniform(0, 3000, n),
                       rng.uniform(0.05, 0.09, n), f(0.09), rng.uniform(0.6, 0.8, n), f(-0.003),
                       np.ones(n, int), {1: 44.0})


def _exact_vs_interpolated(Ts, Ps):
    lines = _hard_lines()
    Q = lambda M, I, T: T ** 1.5
    grid = np.arange(2270.0, 2330.0, 0.005)
    one = lambda T, P: species_cross_section(grid, lines, T, P, 4e-4, Q, wing_cm1=10.0) * 4e-4 * number_density_cm3(T, P)
    exact = np.array([one(T, P) for T, P in zip(Ts, Ps)])
    anchors = choose_anchors(Ts, Ps)
    interp = interpolate_between_anchors(Ts, anchors, exact[anchors])
    core = exact > 1e-3 * exact.max()
    rel = np.max(np.abs(interp - exact)[core] / exact[core])
    dT = np.max(np.abs(np.exp(-interp * 1e4) - np.exp(-exact * 1e4)))      # 100 m path
    return anchors, rel, dT


def test_choose_anchors_spacing_and_ends():
    Ts = np.linspace(216.0, 316.0, 101)
    a = choose_anchors(Ts, np.full(101, 5e4))
    assert a[0] == 0 and a[-1] == 100
    assert np.max(np.diff(Ts[a])) <= 10.0 + 1e-9 and len(a) == 11
    Ts2, Ps2 = isa_troposphere(np.linspace(0, 11000, 101))
    o = np.argsort(Ts2)
    a2 = choose_anchors(Ts2[o], Ps2[o])
    assert np.max(np.abs(np.diff(np.log(Ps2[o][a2])))) <= 0.07 + 1e-9      # pressure spacing respected


def test_interpolated_frames_accuracy_fixed_pressure():
    Ts = np.linspace(216.0, 316.0, 101)
    anchors, rel, dT = _exact_vs_interpolated(Ts, np.full(101, 54019.9))
    assert len(anchors) == 11
    assert rel < 5e-3 and dT < 2e-3, (rel, dT)


def test_interpolated_frames_accuracy_isa_path():
    Ts, Ps = isa_troposphere(np.linspace(0, 11000, 101))
    o = np.argsort(Ts)
    anchors, rel, dT = _exact_vs_interpolated(Ts[o], Ps[o])
    assert rel < 1e-2 and dT < 4e-3, (rel, dT)   # residual dominated by pressure-shift/cut-off-edge effects


def test_anchor_frames_are_returned_unchanged():
    Ts = np.linspace(216.0, 316.0, 21)
    A = np.abs(np.random.default_rng(0).normal(size=(3, 50))) + 1e-3
    out = interpolate_between_anchors(Ts, np.array([0, 10, 20]), A)
    assert np.array_equal(out[[0, 10, 20]], A)
    assert np.all(out > 0)


def test_envelope_keeps_extremes_and_length():
    x = np.linspace(0.0, 10.0, 100000)
    y = np.sin(37 * x) * np.exp(-0.1 * x)
    xs, ys = envelope(x, y, 1500)
    assert len(xs) == 3000 and len(ys) == 3000
    assert ys.min() == y.min() and ys.max() == y.max()
    assert np.all(np.diff(xs) >= 0)


# ------------------------------------------------ wavelength-based quantities
def test_planck_per_um_follows_wien_law_in_wavelength():
    lam = np.linspace(0.5, 10.0, 200000)
    peak = lam[np.argmax(planck_per_um(lam, 1500.0))]
    assert abs(peak - 2897.771955 / 1500.0) < 1e-3          # Wien displacement constant b = 2897.771955 um K


def test_wavelength_and_wavenumber_forms_of_planck_carry_the_same_power():
    nu = np.arange(2250.0, 2350.0, 0.01)
    p_nu = trapezoid(planck_per_cm1(nu, 1500.0), nu)                      # W m^-2 sr^-1
    p_lam = integrate_dlambda_um(nu, planck_per_um(1.0e4 / nu, 1500.0))   # W m^-2 sr^-1
    assert abs(p_nu - p_lam) / p_nu < 1e-6


def test_integral_over_wavelength_gives_the_wavelength_range():
    nu = np.linspace(2250.0, 2350.0, 10001)                      # endpoint included
    area = integrate_dlambda_um(nu, np.ones_like(nu))
    assert abs(area - (1.0e4 / 2250.0 - 1.0e4 / 2350.0)) < 1e-6      # 4.4444 - 4.2553 um


def test_window_mean_of_constant_and_flat_incident_spectrum():
    nu = np.arange(2250.0, 2350.0, 0.01)
    assert abs(window_mean_um(nu, np.full_like(nu, 0.37)) - 0.37) < 1e-12
    I0, unit = make_incident_spectrum(nu, True, 1500.0)
    assert np.all(I0 == 1.0) and "FLAT" in unit

In [14]:
def run_tests():
    names = sorted(n for n, f in globals().items() if n.startswith("test_") and callable(f))
    failed = 0
    for n in names:
        try:
            globals()[n]()
            print("PASS", n)
        except Exception as exc:
            failed += 1
            print("FAIL", n, "->", repr(exc))
    print("\n%d/%d tests passed (synthetic data)" % (len(names) - failed, len(names)))

run_tests()

PASS test_anchor_frames_are_returned_unchanged
PASS test_beer_lambert_consistency_and_bounds
PASS test_c2_value
PASS test_choose_anchors_spacing_and_ends
PASS test_dilution_warning
PASS test_doppler_width_scaling_and_value
PASS test_envelope_keeps_extremes_and_length
PASS test_integral_over_wavelength_gives_the_wavelength_range
PASS test_intensity_scaling_identity_and_formula
PASS test_interpolated_frames_accuracy_fixed_pressure
PASS test_interpolated_frames_accuracy_isa_path
PASS test_isa_troposphere_matches_tabulated_tropopause
PASS test_line_strength_sum_rule
PASS test_loschmidt_number_density
PASS test_low_abundance_and_short_path_limit
PASS test_mixture_additivity_and_linearity_in_mole_fraction
PASS test_optically_thick_limit
PASS test_planck_peak_follows_wien_law_in_wavenumber
PASS test_planck_per_um_follows_wien_law_in_wavelength
PASS test_pressure_and_temperature_dependence_of_lorentz_width
PASS test_resolution_convergence
PASS test_stimulated_emission_rayleigh_jeans_limit
PASS

## 7. Load data and study table

Run `fetch_hitran()` once (or place `.par` files in `data/`), then load.

In [16]:
#fetch_hitran()      # <- uncomment for the first run (needs your HITRAN account/API key set up for HAPI)

In [17]:
CO2, rep_c, files_c = load_linelist("CO2", 2, cfg.DATA_DIR)
N2O, rep_n, files_n = load_linelist("N2O", 4, cfg.DATA_DIR)
print("CO2 lines: %d  | parameters replaced by data-derived estimates: %s" % (len(CO2.nu), rep_c))
print("N2O lines: %d  | parameters replaced by data-derived estimates: %s" % (len(N2O.nu), rep_n))
for msg in dilution_warnings(cfg.X_CO2, cfg.X_N2O):
    print(msg)
print("window: %.0f-%.0f cm-1 = %.4f-%.4f um" % (cfg.NU_MIN, cfg.NU_MAX, 1e4 / cfg.NU_MAX, 1e4 / cfg.NU_MIN))

CO2 lines: 73495  | parameters replaced by data-derived estimates: {'gamma_air': 0, 'n_air': 0, 'gamma_self': 0}
N2O lines: 43399  | parameters replaced by data-derived estimates: {'gamma_air': 0, 'n_air': 0, 'gamma_self': 0}
window: 2250-2350 cm-1 = 4.2553-4.4444 um


In [18]:
# Reference points read from AFGL-TR-86-0110 (Anderson et al. 1986), Tables 1a-1f (scanned copy).
AFGL_REF = [("Tropical, surface",            0, 299.7, 1013.0),
            ("Tropical, cold point",        17, 194.8,   93.7),
            ("Mid-latitude winter, surface", 0, 272.2, 1018.0),
            ("Subarctic winter, surface",    0, 257.2, 1013.0),
            ("U.S. Standard, surface",       0, 288.2, 1013.0),
            ("U.S. Standard, 5 km",          5, 255.7,  540.5),
            ("U.S. Standard, 11 km",        11, 216.8,  227.0)]

def study_table(co2_lines=None, nu_ref=2300.0):
    # Number density, Doppler width and (if lines are loaded) a typical Lorentz width at each reference point.
    gam = nq = None
    if co2_lines is not None:
        sel = (co2_lines.nu >= cfg.NU_MIN) & (co2_lines.nu <= cfg.NU_MAX)
        strong = sel & (co2_lines.S >= 1e-2 * co2_lines.S[sel].max())
        gam, nq = np.median(co2_lines.gamma_air[strong]), np.median(co2_lines.n_air[strong])
    print("%-30s %5s %7s %9s %12s %14s %14s %9s" % ("reference point", "z/km", "T/K", "P/Pa", "n/cm-3",
          "Doppler HWHM", "Lorentz HWHM", "ratio"))
    for name, z, T, p_hpa in AFGL_REF:
        P = p_hpa * 100.0
        gD = float(doppler_hwhm(nu_ref, T, 43.99))
        gL = None if gam is None else float(lorentz_hwhm(gam, gam, nq, T, P, 0.0))
        print("%-30s %5d %7.1f %9.0f %12.3e %14.2e %14s %9s" % (name, z, T, P, number_density_cm3(T, P), gD,
              "n/a (load data)" if gL is None else "%.2e" % gL, "" if gL is None else "%.1f" % (gL / gD)))
    print("Doppler HWHM for CO2 at %.0f cm-1. Lorentz HWHM: median gamma_air of the strong CO2 lines in the window." % nu_ref)
    T1, P1 = isa_troposphere(1000.0)
    print("U.S. Standard 1 km up: dT = %.1f K, dP/P = %.1f %%  -> uniform-slab limit for vertical paths" %
          (288.15 - T1, 100 * (1 - P1 / 101325.0)))

study_table(CO2)

reference point                 z/km     T/K      P/Pa       n/cm-3   Doppler HWHM   Lorentz HWHM     ratio
Tropical, surface                  0   299.7    101300    2.448e+19       2.15e-03       7.30e-02      34.0
Tropical, cold point              17   194.8      9370    3.484e+18       1.73e-03       9.13e-03       5.3
Mid-latitude winter, surface       0   272.2    101800    2.709e+19       2.05e-03       7.85e-02      38.3
Subarctic winter, surface          0   257.2    101300    2.853e+19       1.99e-03       8.13e-02      40.8
U.S. Standard, surface             0   288.2    101300    2.546e+19       2.11e-03       7.51e-02      35.6
U.S. Standard, 5 km                5   255.7     54050    1.531e+19       1.99e-03       4.36e-02      21.9
U.S. Standard, 11 km              11   216.8     22700    7.584e+18       1.83e-03       2.05e-02      11.2
Doppler HWHM for CO2 at 2300 cm-1. Lorentz HWHM: median gamma_air of the strong CO2 lines in the window.
U.S. Standard 1 km up: dT = 6.5

In [19]:
def window_content():
    # How much of each gas's line strength lies in the window? (N2O's nu3 band is centred near 2224 cm-1.)
    print("Sum of line strengths S [cm/molecule at 296 K] and x_i * sum(S) (a rough absorption potential):")
    for lines, x in ((CO2, cfg.X_CO2), (N2O, cfg.X_N2O)):
        inw = (lines.nu >= cfg.NU_MIN) & (lines.nu <= cfg.NU_MAX)
        below = lines.nu < cfg.NU_MIN
        print("  %s: %5d lines in window, sum S = %.3e, x*sum S = %.3e | %d lines below the window (download margin), sum S = %.3e"
              % (lines.name, inw.sum(), lines.S[inw].sum(), x * lines.S[inw].sum(), below.sum(), lines.S[below].sum()))
    print("If N2O's sum S below the window is much larger than inside it, the window cuts through the N2O band: "
          "lower cfg.NU_MIN to see more of it.")

window_content()

Sum of line strengths S [cm/molecule at 296 K] and x_i * sum(S) (a rough absorption potential):
  CO2: 40132 lines in window, sum S = 5.162e-17, x*sum S = 2.184e-20 | 29033 lines below the window (download margin), sum S = 4.805e-20
  N2O: 13129 lines in window, sum S = 1.606e-18, x*sum S = 5.457e-25 | 20792 lines below the window (download margin), sum S = 5.512e-17
If N2O's sum S below the window is much larger than inside it, the window cuts through the N2O band: lower cfg.NU_MIN to see more of it.


## 8. Interactive animation

Set **pressure**, **path length** and a **temperature range** with the sliders, then press **Compute / update**. The notebook computes spectra for that pressure over the
temperature range and shows how the transmittance changes with temperature. After computing, the temperature slider and **Play** run through the frames; the **path length**, the
incident spectrum and the selected wavelength update instantly.

* **101 frames from a few exact calculations.** Only "anchor" frames (about every 10 K) are computed from the line list; the frames in between come from a cubic spline of $\ln\alpha$ versus $T$.
  On deliberately hard synthetic tests the interpolation error in transmittance was about $10^{-5}$ at fixed pressure; the figure title says whether the shown frame is *exact* or *interpolated*, and
  `check_interpolation()` in §9 measures the error on your real data. `cfg.INTERPOLATE_FRAMES = False` makes every frame exact (slow).
* **Pressure and the temperature range need Compute** (line widths and the sweep change). Compute takes tens of seconds to a few minutes.
* **Play** redraws a figure for every frame (about a frame per second); for fluid playback press **Render smooth animation** (about a minute).

*If widgets do not render* (`ipywidgets>=8`, `jupyterlab_widgets`, restart JupyterLab) use `draw(idx, L_in_metres, "blackbody" or "flat", lambda_um)` after the first compute.

In [21]:
# Frame computation, state builder and CSV export (from run_animation.py)
def build_conditions(P_pa, T_lo, T_hi, n_frames):
    """Fixed-pressure sweep: n_frames temperatures from T_lo to T_hi (ascending), constant P."""
    Ts = np.linspace(float(T_lo), float(T_hi), int(n_frames))
    return Ts, np.full(len(Ts), float(P_pa))


def compute_frames(grid, co2, n2o, Ts, Ps, x_co2, x_n2o, partition_fn, interpolate=None, verbose=True):
    """One dict per temperature: T, P, a_co2, a_n2o [cm^-1], exact (True/False).

    With interpolation on (cfg.INTERPOLATE_FRAMES) only the 'anchor' frames are computed
    exactly; frames in between come from a cubic spline of ln(alpha) versus T
    (rt_core.interpolate_between_anchors). Set INTERPOLATE_FRAMES = False for all-exact frames."""
    interpolate = cfg.INTERPOLATE_FRAMES if interpolate is None else interpolate
    n = len(Ts)
    anchors = (choose_anchors(Ts, Ps, cfg.ANCHOR_DT_MAX, cfg.ANCHOR_DLNP_MAX)
               if interpolate else np.arange(n))
    t0 = time.time()
    A_co2, A_n2o = [], []
    for count, i in enumerate(anchors):
        a_co2, a_n2o = absorption_coefficients(
            grid, co2, n2o, Ts[i], Ps[i], x_co2, x_n2o, partition_fn,
            wing_cm1=cfg.WING_CM1, s_min=cfg.S_MIN)
        A_co2.append(a_co2)
        A_n2o.append(a_n2o)
        if verbose:
            print("  exact frame %3d/%d  T = %6.2f K  P = %8.1f Pa   (%.0f s elapsed)"
                  % (count + 1, len(anchors), Ts[i], Ps[i], time.time() - t0))
    a_co2_all = interpolate_between_anchors(Ts, anchors, np.array(A_co2))
    a_n2o_all = interpolate_between_anchors(Ts, anchors, np.array(A_n2o))
    is_exact = np.zeros(n, bool)
    is_exact[anchors] = True
    if verbose:
        print("  %d frames: %d computed exactly, %d interpolated" % (n, len(anchors), n - len(anchors)))
    return [dict(T=Ts[i], P=Ps[i], a_co2=a_co2_all[i], a_n2o=a_n2o_all[i], exact=bool(is_exact[i]))
            for i in range(n)]


def pick_diagnostic_lines(lines, nu_min, nu_max):
    """Strongest line in the window and the highest-E'' line that is still reasonably strong
    (>= 1e-3 of the strongest): they respond to T differently."""
    inside = np.where((lines.nu >= nu_min) & (lines.nu <= nu_max))[0]
    k_strong = inside[np.argmax(lines.S[inside])]
    cand = inside[lines.S[inside] >= 1e-3 * lines.S[k_strong]]
    k_hot = cand[np.argmax(lines.E_low[cand])]
    return int(k_strong), int(k_hot)


def default_lambda_sel_um(co2, n2o, grid):
    """Default selected wavelength [um]: centre of the strongest N2O line in the window
    (falls back to the strongest CO2 line, then to the window centre)."""
    for lines in (n2o, co2):
        inside = (lines.nu >= grid[0]) & (lines.nu <= grid[-1])
        if inside.any():
            return float(1e4 / lines.nu[inside][np.argmax(lines.S[inside])])
    return float(1e4 / grid[len(grid) // 2])


def make_state(grid, co2, n2o, Ts, Ps, x_co2, x_n2o, partition_fn, t_source=None,
               interpolate=None, verbose=True):
    """Compute the frames and bundle everything the dashboard needs into one dict."""
    t_source = cfg.T_SOURCE_K if t_source is None else t_source
    frames = compute_frames(grid, co2, n2o, Ts, Ps, x_co2, x_n2o, partition_fn, interpolate, verbose)
    sample = np.concatenate([f["a_co2"] + f["a_n2o"] for f in frames[::5]])
    sample = sample[sample > 0]
    lo = max(np.percentile(sample, 1.0) / 10.0, 1e-14) if len(sample) else 1e-14
    hi = sample.max() * 3 if len(sample) else 1.0
    diag = []
    for lines in (co2, n2o):
        for k in pick_diagnostic_lines(lines, grid[0], grid[-1]):
            S_T = line_strength_vs_T(lines, k, Ts, partition_fn)
            S296 = line_strength_vs_T(lines, k, [T_REF], partition_fn)[0]
            diag.append(("%s %.4f um, E''=%.0f cm-1" % (lines.name, 1e4 / lines.nu[k], lines.E_low[k]),
                         S_T / S296))
    return dict(frames=frames, Ts=np.asarray(Ts, float), P_pa=float(Ps[0]), x_co2=x_co2, x_n2o=x_n2o,
                ylimD=(lo, hi), diag=diag, t_source=t_source,
                lam_sel_um=default_lambda_sel_um(co2, n2o, grid))


def export_csv(path, grid, frames, L_cm, x_co2, x_n2o, flat=False, t_source=None, notes=""):
    """Long-format CSV: one row per (frame, wavelength). Comment lines start with '#'."""
    t_source = cfg.T_SOURCE_K if t_source is None else t_source
    I0, _unit = make_incident_spectrum(grid, flat, t_source)
    unit = ("arbitrary units (flat, I0 = 1)" if flat
            else "W m^-2 sr^-1 um^-1 (blackbody %.0f K)" % t_source)
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    cols = ["wavelength_um", "wavenumber_cm-1", "T_K", "P_Pa", "I0", "alpha_CO2_cm-1", "alpha_N2O_cm-1",
            "alpha_total_cm-1", "tau", "transmittance", "I_transmitted"]
    with open(path, "w") as fh:
        fh.write("# CO2+N2O transmission model; ILLUSTRATIVE simulation, not a validated product\n")
        fh.write("# x_CO2=%g x_N2O=%g L_cm=%g %s\n" % (x_co2, x_n2o, L_cm, notes))
        fh.write("# I0 unit: %s ; line parameters: HITRAN (see data/data_manifest.json); Voigt profile; Beer-Lambert\n"
                 % unit)
        fh.write(",".join(cols) + "\n")
        for fr in frames:
            tot = fr["a_co2"] + fr["a_n2o"]
            tau, trans, I = beer_lambert(tot, L_cm, I0)
            block = np.column_stack([1.0e4 / grid, grid, np.full_like(grid, fr["T"]), np.full_like(grid, fr["P"]),
                                     I0, fr["a_co2"], fr["a_n2o"], tot, tau, trans, I])
            np.savetxt(fh, block[::-1], delimiter=",", fmt="%.8g")      # ascending wavelength
    print("Wrote", path)

In [22]:
# The figure and the animation builder (from dashboard.py)
class Dashboard:
    N_BINS = 1500

    def __init__(self, grid, state, figsize=(13, 10.5), dpi=80):
        self.grid, self.state = grid, state
        self.Ts = state["Ts"]
        self.lam_min, self.lam_max = 1e4 / grid[-1], 1e4 / grid[0]
        self._w = 1.0e4 / grid ** 2                          # |d lambda / d nu| for wavelength-weighted means
        self._wsum = trapezoid(self._w, grid)
        self.fig = plt.figure(figsize=figsize, dpi=dpi)
        plt.close(self.fig)                                  # stop the inline backend from auto-displaying it
        gs = self.fig.add_gridspec(3, 2, height_ratios=[1.35, 1, 1], left=0.07, right=0.98, top=0.865,
                                   bottom=0.07, hspace=0.55, wspace=0.24)
        self.axT = self.fig.add_subplot(gs[0, :])
        self.axB = self.fig.add_subplot(gs[1, 0])
        self.axD = self.fig.add_subplot(gs[1, 1])
        self.axE1 = self.fig.add_subplot(gs[2, 0])
        self.axE2 = self.fig.add_subplot(gs[2, 1])
        self.title = self.fig.suptitle("", fontsize=9, family="monospace")
        lo, hi = state["ylimD"]
        self.lo = lo
        one = np.ones_like(grid)
        lam = 1e4 / grid

        (self.lCold,) = self.axT.plot(lam, one, color="#7aa6d6", lw=0.7, label="coldest frame")
        (self.lHot,) = self.axT.plot(lam, one, color="#e6a07a", lw=0.7, label="warmest frame")
        (self.lT,) = self.axT.plot(lam, one, color="k", lw=0.9, label="current frame")
        self.vT = self.axT.axvline(lam[0], color="g", ls=":", lw=1)
        self.axT.set_ylim(0, 1.02)
        self.axT.set_ylabel("transmittance (dimensionless)", fontsize=9)
        self.axT.set_title("Transmittance  $\\mathcal{T}(\\lambda)=I/I_0=e^{-\\tau}$  (green dotted line: selected wavelength)",
                           fontsize=10)
        self.axT.legend(fontsize=8, loc="lower left", ncol=3)

        (self.lB0,) = self.axB.plot(lam, one, color="0.65", lw=1.2, label="incident $I_0$ (fixed)")
        (self.lB,) = self.axB.plot(lam, one, color="C3", lw=0.8, label="transmitted $I$")
        self.axB.legend(fontsize=7, loc="lower left")
        self.axB.set_title("Incident and transmitted spectral intensity", fontsize=10)

        (self.lD1,) = self.axD.semilogy(lam, one * lo, color="C1", lw=0.8, label="$\\alpha_{CO_2}$")
        (self.lD2,) = self.axD.semilogy(lam, one * lo, color="C2", lw=0.8, label="$\\alpha_{N_2O}$")
        (self.lD3,) = self.axD.semilogy(lam, one * lo, color="k", lw=0.8,
                                        label="$\\alpha_{total}=\\alpha_{CO_2}+\\alpha_{N_2O}$ (additive)")
        self.vD = self.axD.axvline(lam[0], color="g", ls=":", lw=1)
        self.axD.set_ylim(lo, hi)
        self.axD.set_ylabel("absorption coefficient (cm$^{-1}$)", fontsize=8)
        self.axD.legend(fontsize=7, loc="upper right")
        self.axD.set_title("Absorption coefficients of the two gases", fontsize=10)

        for a in (self.axT, self.axB, self.axD):
            a.set_xlim(self.lam_min, self.lam_max)
            a.set_xlabel("wavelength ($\\mu$m)", fontsize=9)
            a.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
            a.tick_params(labelsize=8)

        (self.lE1s,) = self.axE1.plot(self.Ts, np.ones_like(self.Ts), "o-", ms=2, color="C0",
                                      label="at selected wavelength")
        (self.lE1m,) = self.axE1.plot(self.Ts, np.ones_like(self.Ts), "o-", ms=2, color="C3",
                                      label="window mean (wavelength-weighted)")
        self.vE1 = self.axE1.axvline(self.Ts[0], color="k", lw=1)
        self.axE1.set_ylim(0, 1.02)
        self.axE1.set_ylabel("transmittance", fontsize=9)
        self.axE1.legend(fontsize=7, loc="lower right")
        self.axE1.set_title("Transmittance versus gas temperature", fontsize=10)

        for name, ratio in state["diag"]:
            self.axE2.semilogy(self.Ts, ratio, "o-", ms=2, label=name)
        self.vE2 = self.axE2.axvline(self.Ts[0], color="k", lw=1)
        self.axE2.set_ylabel("S(T) / S(296 K)", fontsize=9)
        self.axE2.legend(fontsize=6)
        self.axE2.set_title("Line strength vs T: different lines, different trends", fontsize=10)
        for a in (self.axE1, self.axE2):
            a.set_xlabel("gas temperature T (K)", fontsize=9)
            a.tick_params(labelsize=8)
        self._key = None

    # ------------------------------------------------------------------
    def _set(self, line, y):
        xe, ye = envelope(self.grid, y, self.N_BINS)
        line.set_data(1e4 / xe, ye)

    def _trans(self, frame, L_cm):
        return np.exp(-(frame["a_co2"] + frame["a_n2o"]) * L_cm)

    def _cache(self, L_cm, i_sel):
        """Transmittance at the selected wavelength and window mean for every frame (recomputed
        only when the path length or the selected wavelength changes)."""
        if self._key == (L_cm, i_sel):
            return
        frames = self.state["frames"]
        self.t_sel = np.empty(len(frames))
        self.t_mean = np.empty(len(frames))
        for k, f in enumerate(frames):
            tr = self._trans(f, L_cm)
            self.t_sel[k] = tr[i_sel]
            self.t_mean[k] = trapezoid(tr * self._w, self.grid) / self._wsum
        self.lE1s.set_ydata(self.t_sel)
        self.lE1m.set_ydata(self.t_mean)
        self._key = (L_cm, i_sel)

    def update(self, idx, logL, src, lam_sel_um):
        """Show frame `idx`, path length 10**logL metres, source 'blackbody' or 'flat',
        selected wavelength lam_sel_um (micrometres)."""
        st = self.state
        fr = st["frames"][idx]
        L_cm = 10 ** logL * 100.0
        i_sel = int(np.argmin(np.abs(self.grid - 1e4 / lam_sel_um)))
        I0, unit = make_incident_spectrum(self.grid, src == "flat", st["t_source"])
        tot = fr["a_co2"] + fr["a_n2o"]
        tau, trans, I = beer_lambert(tot, L_cm, I0)
        self._cache(L_cm, i_sel)

        self._set(self.lCold, self._trans(st["frames"][0], L_cm))
        self._set(self.lHot, self._trans(st["frames"][-1], L_cm))
        self._set(self.lT, trans)
        self.axT.legend(handles=[self.lCold, self.lHot, self.lT],
                        labels=["coldest frame (%.0f K)" % st["Ts"][0], "warmest frame (%.0f K)" % st["Ts"][-1],
                                "current frame (%.1f K)" % fr["T"]], fontsize=8, loc="lower left", ncol=3)
        self._set(self.lB0, I0)
        self._set(self.lB, I)
        self.axB.set_ylim(0, 1.05 * I0.max())
        self.axB.set_ylabel(unit, fontsize=8)
        self._set(self.lD1, np.maximum(fr["a_co2"], self.lo))
        self._set(self.lD2, np.maximum(fr["a_n2o"], self.lo))
        self._set(self.lD3, np.maximum(tot, self.lo))
        lam_marker = 1e4 / self.grid[i_sel]
        for v in (self.vT, self.vD):
            v.set_xdata([lam_marker, lam_marker])
        for v in (self.vE1, self.vE2):
            v.set_xdata([fr["T"], fr["T"]])
        W_nm = 1000.0 * integrate_dlambda_um(self.grid, 1.0 - trans)
        self.title.set_text(
            "CO2 + N2O slab, %.4f-%.4f um (%.0f-%.0f cm-1)   [ILLUSTRATIVE settings, HITRAN line data]\n"
            "T = %.1f K (%s frame)   P = %.0f Pa   L = %.3g m   x_CO2 = %.2f ppm   x_N2O = %.1f ppb\n"
            "selected %.4f um: tau = %.3e, transmittance = %.3e   window-mean = %.4f   equivalent width = %.3f nm (flat-I0 measure)"
            % (self.lam_min, self.lam_max, self.grid[0], self.grid[-1], fr["T"],
               "exact" if fr.get("exact", True) else "interpolated", fr["P"], L_cm / 100.0,
               st["x_co2"] * 1e6, st["x_n2o"] * 1e9, lam_marker, tau[i_sel], trans[i_sel],
               self.t_mean[idx], W_nm))


def build_animation(grid, state, L_m, src="blackbody", lam_sel_um=None, stride=1, fps=15, dpi=65):
    """matplotlib FuncAnimation over all frames (every `stride`-th). In a notebook:
         HTML(anim.to_jshtml(fps=15))
    or save with anim.save('x.gif', writer='pillow', fps=15) / anim.save('x.mp4', writer='ffmpeg')."""
    import matplotlib.animation as animation
    lam_sel_um = state["lam_sel_um"] if lam_sel_um is None else lam_sel_um
    d = Dashboard(grid, state, figsize=(12.5, 10), dpi=dpi)
    idxs = list(range(0, len(state["Ts"]), stride))

    def _frame(k):
        d.update(idxs[k], np.log10(L_m), src, lam_sel_um)
        return []

    return animation.FuncAnimation(d.fig, _frame, frames=len(idxs), interval=1000.0 / fps,
                                   blit=False, repeat=True)

In [23]:
GRID = np.arange(cfg.NU_MIN, cfg.NU_MAX + 0.5 * cfg.NU_STEP, cfg.NU_STEP)
ST = {}                                    # the current computed state (frames, temperatures, ...)

def draw(idx, L_m, src, lam_um):
    # Static fallback (no widgets needed): one figure for frame idx, path length L_m metres.
    d = Dashboard(GRID, ST)
    d.update(idx, np.log10(L_m), src, lam_um)
    display(d.fig)

In [24]:
# ---------------------------- controls (SI units) ----------------------------
sty = {"description_width": "110px"}
wide = widgets.Layout(width="520px")
P_sl = widgets.FloatSlider(value=cfg.P_DEFAULT, min=cfg.P_BOUNDS[0], max=cfg.P_BOUNDS[1], step=cfg.P_STEP,
                           description="Pressure P (Pa)", readout_format=".0f", continuous_update=False, style=sty, layout=wide)
L_sl = widgets.FloatLogSlider(value=cfg.L_DEFAULT_M, base=10, min=cfg.L_BOUNDS_LOG10_M[0], max=cfg.L_BOUNDS_LOG10_M[1],
                              step=0.05, description="Path length L (m)", readout_format=".3g", continuous_update=False,
                              style=sty, layout=wide)
T_rng = widgets.FloatRangeSlider(value=cfg.T_DEFAULT, min=cfg.T_BOUNDS[0], max=cfg.T_BOUNDS[1], step=1.0,
                                 description="T range (K)", readout_format=".0f", continuous_update=False, style=sty, layout=wide)
n_sl = widgets.IntSlider(value=cfg.N_FRAMES, min=11, max=201, step=10, description="Frames", continuous_update=False,
                         style=sty, layout=wide)
xc_box = widgets.FloatText(value=round(cfg.X_CO2 * 1e6, 2), description="CO2 (ppm)", style=sty, layout=widgets.Layout(width="200px"))
xn_box = widgets.FloatText(value=round(cfg.X_N2O * 1e9, 2), description="N2O (ppb)", style=sty, layout=widgets.Layout(width="200px"))
src_dd = widgets.Dropdown(options=[("blackbody %.0f K" % cfg.T_SOURCE_K, "blackbody"), ("flat (I0 = 1)", "flat")],
                          value="blackbody", description="Incident I0", style=sty, layout=widgets.Layout(width="260px"))
lam_box = widgets.FloatText(value=round(1e4 / cfg.NU_MAX, 4), step=0.001, description="Selected λ (µm)", style=sty,
                            layout=widgets.Layout(width="230px"))
btn_compute = widgets.Button(description="Compute / update", button_style="primary", layout=widgets.Layout(width="190px"))
btn_smooth = widgets.Button(description="Render smooth animation", layout=widgets.Layout(width="210px"))
btn_csv = widgets.Button(description="Export CSV (current frame)", layout=widgets.Layout(width="210px"))
play = widgets.Play(min=0, max=0, step=1, interval=250, value=0, description="Play")
T_slider = widgets.SelectionSlider(options=[("-", 0)], index=0, description="Temperature", continuous_update=False,
                                   style=sty, layout=widgets.Layout(width="620px"))
out_status = widgets.Output()
out_fig = widgets.Output()
widgets.link((play, "value"), (T_slider, "index"))

DASH = {"obj": None}
BUSY = {"flag": False}

def get_dash():
    if DASH["obj"] is None:
        DASH["obj"] = Dashboard(GRID, ST)
    return DASH["obj"]

def redraw(*_):
    if not ST or BUSY["flag"]:
        return
    with out_fig:
        clear_output(wait=True)
        d = get_dash()
        d.update(T_slider.index, np.log10(L_sl.value), src_dd.value, lam_box.value)
        display(d.fig)

def on_compute(_):
    BUSY["flag"] = True
    try:
        with out_status:
            clear_output()
            x_co2, x_n2o = xc_box.value * 1e-6, xn_box.value * 1e-9
            for msg in dilution_warnings(x_co2, x_n2o):
                print(msg)
            lo, hi = T_rng.value
            Ts, Ps = build_conditions(P_sl.value, lo, hi, n_sl.value)
            print("Computing %d frames: P = %.0f Pa, %.0f-%.0f K, %d grid points ..." % (len(Ts), P_sl.value, lo, hi, len(GRID)))
            ST.clear()
            ST.update(make_state(GRID, CO2, N2O, Ts, Ps, x_co2, x_n2o, hapi_partition_fn))
            print("done.")
        DASH["obj"] = None
        play.max = len(ST["Ts"]) - 1
        T_slider.options = [("%.1f K" % T, i) for i, T in enumerate(ST["Ts"])]
        T_slider.index = len(ST["Ts"]) // 2
        lam_box.value = round(ST["lam_sel_um"], 4)
    finally:
        BUSY["flag"] = False
    redraw()

def on_smooth(_):
    if not ST:
        return
    with out_status:
        clear_output()
        print("Rendering %d frames ... (about a minute)" % len(ST["Ts"]))
        anim = build_animation(GRID, ST, L_sl.value, src_dd.value, lam_box.value, stride=1)
        html = anim.to_jshtml(fps=15)
        print("done.")
    with out_fig:
        clear_output(wait=True)
        display(HTML(html))

def on_csv(_):
    if not ST:
        return
    fr = ST["frames"][T_slider.index]
    L_cm = L_sl.value * 100.0
    path = os.path.join("outputs", "spectrum_T%.1fK_P%.0fPa_L%.0fm.csv" % (fr["T"], fr["P"], L_sl.value))
    with out_status:
        export_csv(path, GRID, [fr], L_cm, ST["x_co2"], ST["x_n2o"], src_dd.value == "flat", cfg.T_SOURCE_K)

T_slider.observe(redraw, names="index")
for w in (L_sl, src_dd, lam_box):
    w.observe(redraw, names="value")
btn_compute.on_click(on_compute)
btn_smooth.on_click(on_smooth)
btn_csv.on_click(on_csv)

display(widgets.VBox([
    widgets.HTML("<b>Conditions</b> &nbsp; (SI units; pressure and temperature range need <i>Compute</i>, the rest update instantly)"),
    P_sl, L_sl, T_rng, n_sl,
    widgets.HBox([xc_box, xn_box, src_dd, lam_box]),
    widgets.HBox([btn_compute, btn_smooth, btn_csv]),
    out_status,
    widgets.HTML("<b>Animation</b>"),
    widgets.HBox([play, T_slider]),
    out_fig]))
on_compute(None)                            # first computation with the default settings

### Save the smooth animation as a file

`build_animation` renders every frame once. GIF needs Pillow; MP4 needs ffmpeg. The embedded player adds roughly 20 MB to the notebook for 101 frames when saved with outputs: use `stride=2` or clear outputs before sharing it.

In [26]:
# anim = build_animation(GRID, ST, L_sl.value, src_dd.value, lam_box.value, stride=2)
# anim.save("co2_n2o_transmission.gif", writer="pillow", fps=15)
# anim.save("co2_n2o_transmission.mp4", writer="ffmpeg", fps=15, dpi=100)

## 9. Checks you should run on real data

These close the gaps listed in the status note. None of them has been run on real HITRAN data by me.

1. **Window and band content** – `window_content()` (§7) and `strongest_lines()` below: does the window contain enough N₂O to be useful?
2. **Interpolation error** – `check_interpolation()`.
3. **Sensitivity** to the line-wing cut-off, weak-line threshold and grid step – `sensitivity()`.
4. **Cross-check against HAPI's own Voigt routine** – `check_against_hapi()`.
5. **Real-data sanity**: with a long path CO₂ should be black near its band centre (about 4.26 µm); with a very short path $\mathcal T\to1$ everywhere.

In [28]:
def strongest_lines(lines, n=8):
    sel = np.where((lines.nu >= cfg.NU_MIN) & (lines.nu <= cfg.NU_MAX))[0]
    top = sel[np.argsort(lines.S[sel])[::-1][:n]]
    print(lines.name, ": %d lines in the window" % len(sel))
    for k in top:
        print("  lambda = %.5f um (%.4f cm-1)   S = %.3e   E'' = %8.2f   gamma_air = %.4f   iso %d"
              % (1e4 / lines.nu[k], lines.nu[k], lines.S[k], lines.E_low[k], lines.gamma_air[k], lines.iso[k]))

strongest_lines(CO2); strongest_lines(N2O)

CO2 : 40132 lines in the window
  lambda = 4.28097 um (2335.9194 cm-1)   S = 3.330e-18   E'' =   106.13   gamma_air = 0.0742   iso 1
  lambda = 4.28420 um (2334.1563 cm-1)   S = 3.281e-18   E'' =   133.44   gamma_air = 0.0731   iso 1
  lambda = 4.27779 um (2337.6580 cm-1)   S = 3.278e-18   E'' =    81.94   gamma_air = 0.0755   iso 1
  lambda = 4.28749 um (2332.3689 cm-1)   S = 3.143e-18   E'' =   163.87   gamma_air = 0.0722   iso 1
  lambda = 4.27465 um (2339.3722 cm-1)   S = 3.113e-18   E'' =    60.87   gamma_air = 0.0770   iso 1
  lambda = 4.29082 um (2330.5571 cm-1)   S = 2.937e-18   E'' =   197.42   gamma_air = 0.0713   iso 1
  lambda = 4.27157 um (2341.0620 cm-1)   S = 2.831e-18   E'' =    42.92   gamma_air = 0.0787   iso 1
  lambda = 4.29420 um (2328.7209 cm-1)   S = 2.680e-18   E'' =   234.08   gamma_air = 0.0706   iso 1
N2O : 13129 lines in the window
  lambda = 4.44356 um (2250.4492 cm-1)   S = 2.231e-19   E'' =   588.78   gamma_air = 0.0694   iso 1
  lambda = 4.44244 um (2251

In [29]:
def check_interpolation(n_check=3):
    # Compare interpolated frames with fully computed ones on YOUR data (a few non-anchor frames).
    frames, Ts = ST["frames"], ST["Ts"]
    L_cm = L_sl.value * 100.0
    nonexact = [i for i, f in enumerate(frames) if not f["exact"]]
    if not nonexact:
        print("All frames were computed exactly.")
        return
    for k in np.linspace(0, len(nonexact) - 1, n_check).astype(int):
        i = nonexact[k]
        a1, a2 = absorption_coefficients(GRID, CO2, N2O, Ts[i], frames[i]["P"], ST["x_co2"], ST["x_n2o"],
                                         hapi_partition_fn, wing_cm1=cfg.WING_CM1, s_min=cfg.S_MIN)
        for name, ex, ip in (("CO2", a1, frames[i]["a_co2"]), ("N2O", a2, frames[i]["a_n2o"])):
            core = ex > 1e-3 * ex.max()
            rel = np.max(np.abs(ip - ex)[core] / ex[core])
            dT = np.max(np.abs(np.exp(-ip * L_cm) - np.exp(-ex * L_cm)))
            print("frame %3d (T = %.1f K) %s: max rel. error of alpha (core) = %.2e, max |d transmittance| at L = %.3g m: %.2e"
                  % (i, Ts[i], name, rel, L_cm / 100, dT))

# check_interpolation()     # computes 2 x n_check exact frames: a minute or two

In [30]:
def sensitivity(window=None, T=None, P_pa=None, L_m=None):
    # How much do the numerical approximations matter inside `window` (cm-1)? A narrower window is faster.
    window = (cfg.NU_MIN, cfg.NU_MIN + 30.0) if window is None else window
    T = float(np.mean(ST["Ts"])) if T is None else T
    P_pa = ST["P_pa"] if P_pa is None else P_pa
    L_cm = (L_sl.value if L_m is None else L_m) * 100.0
    base = dict(step=cfg.NU_STEP, wing=cfg.WING_CM1, smin=cfg.S_MIN)
    variants = [("baseline", {}),
                ("wing 5 cm-1", dict(wing=5.0)), ("wing 25 cm-1", dict(wing=25.0)), ("wing 50 cm-1", dict(wing=50.0)),
                ("S_min x100", dict(smin=cfg.S_MIN * 100)), ("S_min /100", dict(smin=cfg.S_MIN / 100)),
                ("step x2", dict(step=cfg.NU_STEP * 2)), ("step /2", dict(step=cfg.NU_STEP / 2))]
    print("window %.0f-%.0f cm-1, T = %.1f K, P = %.0f Pa, L = %.3g m" % (window + (T, P_pa, L_cm / 100)))
    print("%-14s %16s %14s %14s" % ("variant", "int tau dlam/nm", "mean T", "rel. change"))
    ref = None
    for name, ch in variants:
        p = dict(base, **ch)
        g = np.arange(window[0], window[1], p["step"])
        a1, a2 = absorption_coefficients(g, CO2, N2O, T, P_pa, ST["x_co2"], ST["x_n2o"], hapi_partition_fn,
                                         wing_cm1=p["wing"], s_min=p["smin"])
        itau = 1000.0 * integrate_dlambda_um(g, (a1 + a2) * L_cm)
        mT = window_mean_um(g, np.exp(-(a1 + a2) * L_cm))
        ref = itau if ref is None else ref
        print("%-14s %16.6e %14.6e %14.3e" % (name, itau, mT, (itau - ref) / ref))

# sensitivity()      # takes a few minutes; shrink `window` for speed

In [31]:
def check_against_hapi(name="N2O", nu_a=2250.0, nu_b=2260.0, T_list=(296.0, 255.65), P_atm=0.533, x_self=3.4e-7):
    # Compare our cross section with HAPI's absorptionCoefficient_Voigt for the same lines (needs data/ files).
    # Cores should agree closely at 296 K; far wings and T != 296 K can differ because of cut-off and exponent conventions.
    import hapi
    M = dict(CO2=2, N2O=4)[name]
    hapi.db_begin(cfg.DATA_DIR)
    table = [os.path.splitext(os.path.basename(p))[0]
             for p in glob.glob(os.path.join(cfg.DATA_DIR, name + "*.data")) + glob.glob(os.path.join(cfg.DATA_DIR, name + "*.par"))][0]
    lines = CO2 if name == "CO2" else N2O
    g = np.arange(nu_a, nu_b, cfg.NU_STEP)
    for T in T_list:
        mine = species_cross_section(g, lines, T, P_atm * ATM_PA, x_self, hapi_partition_fn, wing_cm1=cfg.WING_CM1, s_min=0.0)
        nu_h, sig_h = hapi.absorptionCoefficient_Voigt(SourceTables=[table], Environment={"T": T, "p": P_atm},
                       WavenumberRange=(nu_a, nu_b), WavenumberStep=cfg.NU_STEP, HITRAN_units=True,
                       Diluent={"air": 1.0 - x_self, "self": x_self})
        ref = np.interp(g, nu_h, sig_h)
        core = mine > 0.2 * mine.max()
        rel = np.abs(mine[core] - ref[core]) / ref[core]
        print("%s T=%.2f K: core points %d, max |rel diff| = %.3e, median = %.3e" % (name, T, core.sum(), rel.max(), np.median(rel)))

# check_against_hapi()   # check HAPI's signature first: hapi.getHelp(hapi.absorptionCoefficient_Voigt)

## 10. Export

The **Export CSV** button writes the current frame. To write *all* frames in one long CSV (wavelength ascending):

In [33]:
# export_csv("outputs/all_frames.csv", GRID, ST["frames"], L_sl.value * 100.0, ST["x_co2"], ST["x_n2o"], src_dd.value == "flat", cfg.T_SOURCE_K)

## 11. References (what each source supports)

* Anderson, Clough, Kneizys, Chetwynd, Shettle, *AFGL Atmospheric Constituent Profiles (0–120 km)*, AFGL-TR-86-0110, Environmental Research Papers No. 954 (1986), https://apps.dtic.mil/sti/pdfs/ADA175173.pdf —
  the six reference atmospheres used for the layer choice and the temperature/pressure ranges; CO₂ and N₂O mixing-ratio profiles (the 1986 mixing ratios themselves are out of date, current values are from NOAA below).
* *U.S. Standard Atmosphere, 1976*, U.S. Government Printing Office, Washington, DC — $T_0=288.15$ K, $P_0=101325$ Pa, 6.5 K/km lapse rate, tropopause 11 km (216.65 K, 22 632 Pa).
* NOAA GML, CO₂ trends (423.03 ppm, Sep 2025): https://gml.noaa.gov/ccgg/trends/global.html · N₂O trends (339.76 ppb, Mar 2026): https://www.gml.noaa.gov/ccgg/trends_n2o
* HITRAN, *Definitions and units*, https://hitran.org/docs/definitions-and-units/ — units of $S$, $\gamma$ (HWHM at 296 K and 1 atm), $n_{air}$, and the temperature scaling of $S$.
* Kochanov et al., "HITRAN Application Programming Interface (HAPI)…", JQSRT **177**, 15–30 (2016), doi:10.1016/j.jqsrt.2016.03.005; https://hitran.org/hapi/ — HAPI; key requirement and daily request limit.
* Gordon et al., "The HITRAN2024 molecular spectroscopic database", JQSRT **353**, 109807 (2026), doi:10.1016/j.jqsrt.2026.109807 — database edition (as listed on the HAPI page).
* Gamache et al., "Partition sums for molecules and their isotopologues for HITRAN2024", JQSRT **345**, 109568 (2025), doi:10.1016/j.jqsrt.2025.109568 — TIPS partition sums used by HAPI.
* Hargreaves et al., "Spectroscopic line parameters of NO, NO₂, and N₂O for the HITEMP database", JQSRT **232**, 35–53 (2019) — high-temperature N₂O (¹⁴N₂¹⁶O only).
* Hashemi et al., HITRAN2020 air/self-broadening updates for N₂O and CO, and "Revising the line-shape parameters for air- and self-broadened CO₂ lines toward a sub-percent accuracy level" (2020) — origin of the broadening parameters (journal details not verified here).
* Doppler width, the Voigt profile via the Faddeeva function (SciPy `wofz`), Beer–Lambert law, and the Planck function are standard results used without a verified equation number. **No equation or page numbers are claimed.**
* The position of the N₂O ν₃ band (centre near 2224 cm⁻¹, mostly below about 2260 cm⁻¹) is quoted from the spectroscopy literature without a verified page reference; `window_content()` checks it against your own line list.